# Instruction-following: can the model copy what it is given?

**The gate.** `thought01.md` proposes teaching the model to *repeat* (`Say: {s_e}` / `Nói: {s_v}`) before
any mapping is trained. If the model already copies zero-shot, fine-tuning is unnecessary. If it does not,
the copy task is the first thing to fine-tune.

Number of sentences: **2,000** (deterministic sample of the MTET test split, seed 42).
Each model is prompted **in its own language**.

In [ ]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 1. The exact prompt strings

In [ ]:
from src import copy_eval

rows = copy_eval.sample_sentences(6)
for r in rows[:6]:
    print("vi:", r["vi"][:90])
    print("en:", r["en"][:90])
    print("   prompt(vi model):", C.MODELS["vit5-base"]["copy_prefix"] + r["vi"][:60])
    print("   prompt(en model):", C.MODELS["flan-t5-base"]["copy_prefix"] + r["en"][:60])
    print()

## 2. Live sample — 8 prompts per base model

Only 8, so the cell stays fast. Section 3 shows the full 2,000-row result.

In [ ]:
def demo(key, n=8):
    spec = C.MODELS[key]
    lang = spec["lang"]
    tok, model, _ = M.load(key)
    rs = copy_eval.sample_sentences(n)
    prompts = [spec["copy_prefix"] + r[lang] for r in rs]
    hyps = copy_eval._generate(tok, model, prompts, 96)
    for src, hyp in zip([r[lang] for r in rs], hyps):
        ok = "OK " if hyp.strip() == src.strip() else "BAD"
        print(f"{ok} src: {src[:80]}")
        print(f"    hyp: {hyp[:80]}")
    M.free(model)

demo("vit5-base")
demo("flan-t5-base")

## 3. Full result, 2,000 sentences x 4 models

In [ ]:
table = read_csv("copy_summary.csv")
cols = ["model", "size_class", "n", "exact_match", "exact_match_norm", "bleu", "chrf", "tok_recall", "mean_hyp_words", "mean_ref_words"]
if table:
    print(" | ".join(cols))
    for r in table:
        print(" | ".join(str(r[c]) for c in cols))

## 4. Failure taxonomy

Four failure shapes matter, because they need different fixes: **empty**, **truncated**, **looped**, **paraphrased**.

In [ ]:
def classify(src, hyp):
    if not hyp.strip():
        return "empty"
    if hyp.strip() == src.strip():
        return "exact"
    if len(hyp.split()) < 0.8 * len(src.split()):
        return "truncated"
    if len(hyp.split()) > 1.8 * len(src.split()):
        return "looped"
    return "paraphrased"

for r in read_csv("copy_summary.csv"):
    key = r["model"]
    rows = read_csv(f"copy_{key}.csv")
    if not rows:
        continue
    from collections import Counter
    cnt = Counter(classify(x["src"], x["hyp"]) for x in rows)
    total = sum(cnt.values())
    print(key, {k: f"{v/total:.1%}" for k, v in cnt.most_common()})

## 5. Verdict

In [ ]:
for r in read_csv("copy_summary.csv"):
    verdict = "copies" if float(r["exact_match_norm"]) > 0.5 else "does not copy"
    print(f"{r['model']:<14} {verdict:<12} exact {float(r['exact_match_norm']):.1%}  BLEU {r['bleu']}")

**What to conclude.** `exact_match_norm` is the number that decides whether fine-tuning on the copy task
is needed. BLEU on the copy task is a softer version of the same signal — it rewards partial copying that
exact match punishes, which is why both are reported.

If a model copies its own language reliably, the next experiment is the *mapping* (`small-experiment`),
not more fine-tuning.